# UpliftBench: cross-estimator comparison

Loads `artifacts/leaderboard.parquet` and `artifacts/scored_sample.parquet`
to produce the five plots used in the blog post:

1. Qini curve per estimator.
2. AUUC bar chart.
3. Segment-size breakdown.
4. DoWhy refutation results.
5. Persuadable share vs budget.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from upliftbench.config import (
    DOWHY_REFUTATION_JSON,
    LEADERBOARD_PARQUET,
    OUTCOME_VISIT,
    SCORED_SAMPLE_PARQUET,
    TREATMENT_COL,
)
from upliftbench.eval.qini import qini_curve
from upliftbench.segmentation import ALL_SEGMENTS, budget_allocation

In [ ]:
leaderboard = pd.read_parquet(LEADERBOARD_PARQUET)
scored = pd.read_parquet(SCORED_SAMPLE_PARQUET)
refute = json.loads(Path(DOWHY_REFUTATION_JSON).read_text())
leaderboard.sort_values("qini", ascending=False)

## 1. Qini curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
t = scored[TREATMENT_COL].to_numpy()
y = scored[OUTCOME_VISIT].to_numpy()
for col in [c for c in scored.columns if c.startswith("pred_cate_") and c != "pred_cate_best"]:
    name = col.removeprefix("pred_cate_").replace("_", "-")
    xs, ys = qini_curve(t, y, scored[col].to_numpy())
    ax.plot(xs, ys, label=name)
ax.plot([0, 1], [0, ys[-1]], color="gray", linestyle="--", label="random")
ax.set_xlabel("Population fraction")
ax.set_ylabel("Cumulative uplift")
ax.set_title("Qini curves")
ax.legend()
fig.savefig("../artifacts/plots/qini_curves.png", dpi=120, bbox_inches="tight")
fig

## 2. AUUC bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
lb = leaderboard.sort_values("auuc", ascending=False)
ax.bar(lb["estimator"], lb["auuc"], color="steelblue")
ax.set_ylabel("AUUC")
ax.set_title("AUUC per estimator")
ax.tick_params(axis="x", rotation=20)
fig.savefig("../artifacts/plots/auuc_bars.png", dpi=120, bbox_inches="tight")
fig

## 3. Segment-size breakdown

In [ ]:
counts = pd.Series(scored["segment"]).value_counts().reindex(ALL_SEGMENTS).fillna(0)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(counts.index, counts.values, color=["#2ecc71", "#3498db", "#7f8c8d", "#e74c3c"])
ax.set_title("Segment size in held-out sample")
ax.set_ylabel("count")
fig.savefig("../artifacts/plots/segment_sizes.png", dpi=120, bbox_inches="tight")
fig

## 4. DoWhy refutation

In [ ]:
rows = []
for name, info in refute["refutations"].items():
    rows.append(
        {
            "refuter": name,
            "new_estimate": info.get("new_estimate", np.nan),
            "p_value": info.get("p_value", np.nan),
        }
    )
ref_df = pd.DataFrame(rows)
print(f"Point estimate (ATE): {refute['estimate']:+.6f}")
ref_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.axvline(refute["estimate"], color="black", linestyle="--", label="original estimate")
ax.barh(ref_df["refuter"], ref_df["new_estimate"], color="steelblue")
ax.set_xlabel("Refuted estimate")
ax.set_title("DoWhy refutation (lower = closer to zero)")
ax.legend()
fig.savefig("../artifacts/plots/refutation.png", dpi=120, bbox_inches="tight")
fig

## 5. Persuadable share vs budget

In [ ]:
df = scored[["pred_cate_best", "segment"]].rename(columns={"pred_cate_best": "pred_cate"})
budgets = np.linspace(0.01, 1.0, 50)
shares = []
for b in budgets:
    a = budget_allocation(df, budget_pct=float(b))
    p = a["counts"]["persuadable"] / max(a["n_targeted"], 1)
    shares.append(p)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(budgets * 100, shares, color="#2ecc71")
ax.set_xlabel("Treatment budget (% of pop)")
ax.set_ylabel("Persuadable share of targeted")
ax.set_title("Persuadable fraction by budget")
fig.savefig("../artifacts/plots/persuadable_curve.png", dpi=120, bbox_inches="tight")
fig